# Week 2: binary classification of movie reviews (IMDb)

*Deep Learning for Engineering, Durham University, Michaelmas Term 2026*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimageramiseresht/ENGI-48915---Deep-Learning-for-Engineering/blob/main/week2/imdb_binary.ipynb)

Is a movie review **positive or negative**? This notebook follows the IMDb slides in Sections 3 and 6 of the lecture, and Part 2 follows Sections 7 and 8. The engineering analogue is a normal-or-fault decision from a sensor record.

The pipeline is the MNIST one with four changes:

| | MNIST | IMDb |
|---|---|---|
| input | 784 pixel values | 10,000-long bag-of-words vector |
| last layer | `Dense(10, activation="softmax")` | `Dense(1, activation="sigmoid")` |
| loss | `sparse_categorical_crossentropy` | `binary_crossentropy` |
| prediction | argmax of 10 probabilities | probability above a threshold |

Run the cells from top to bottom with **Shift + Enter**. Everything runs on a laptop CPU in a few minutes.

## Step 0: setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, regularizers

keras.utils.set_random_seed(42)
print(keras.__version__, keras.backend.backend())

## Step 1: load the reviews and look at them

25,000 training and 25,000 test reviews, exactly balanced. Keras delivers each review as a **list of word indices**, keeping only the 10,000 most frequent words. Index 1 marks the start of a review and 2 an unknown word; real words start at 3.

In [ ]:
(seq_train, yb_train), (seq_test, yb_test) = \
    keras.datasets.imdb.load_data(num_words=10000)

print(len(seq_train), len(seq_test))   # 25000 25000
print(np.bincount(yb_train))           # [12500 12500]: balanced
print(seq_train[0][:12])               # a review is a list of word indices

lengths = [len(s) for s in seq_train]
print("words per review: min", min(lengths), "median", int(np.median(lengths)), "max", max(lengths))
plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=80)
plt.xlabel("words per review"); plt.ylabel("reviews")
plt.show()

Turn the indices back into words to read a review. Keras's word list starts at 1, and the review indices are offset by 3.

In [ ]:
word_index = keras.datasets.imdb.get_word_index()       # word -> index
index_word = {i + 3: w for w, i in word_index.items()}  # undo the offset of 3
index_word.update({0: "<pad>", 1: "<start>", 2: "<unknown>"})

def decode(seq):
    return " ".join(index_word.get(i, "?") for i in seq)

for i in range(2):
    print("positive" if yb_train[i] else "negative", "|", decode(seq_train[i])[:400], "...\n")

**Question:** one review has 10 words, the next has 2,000. Why is that a problem for a Dense layer?

<details><summary>Show the answer</summary>

A Dense layer has a fixed number of weights per unit, one per input, so every sample must have the same number of inputs. The bag of words in Step 2 turns any review into exactly 10,000 numbers (Section 3).

</details>

## Step 2: bag of words

Position *i* of a review's vector is 1 if word *i* appears in the review, otherwise 0 (**multi-hot**). Every review becomes 10,000 numbers, whatever its length, and word order is lost.

In [ ]:
def multi_hot(sequences, dim=10000):
    out = np.zeros((len(sequences), dim), dtype="float32")
    for i, seq in enumerate(sequences):
        out[i, seq] = 1.0          # word i present: 1
    return out

xb_train = multi_hot(seq_train)
xb_test = multi_hot(seq_test)
print(xb_train.shape)              # (25000, 10000)
print(f"{xb_train.mean():.2%} of the entries are 1")
print(f"{xb_train.nbytes / 1e9:.1f} GB for the training set alone")
yb_train = yb_train.astype("float32")
yb_test = yb_test.astype("float32")

## Step 3: build the model

Two hidden layers of 16 ReLU units, then **one sigmoid unit**: the probability that the review is positive (Section 2).

**Question:** how many parameters?

<details><summary>Show the answer</summary>

10,000 × 16 + 16 = 160,016 for the first layer, 16 × 16 + 16 = 272 for the second, and 16 + 1 = 17 for the output: **160,305** in total. Almost all of them sit in the first layer, one weight per word per unit.

</details>

In [ ]:
model_b = keras.Sequential([
    keras.Input(shape=(10000,)),            # was 784
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),  # was Dense(10, softmax)
])
model_b.compile(optimizer="adam",
                loss="binary_crossentropy",  # was sparse CCE
                metrics=["accuracy"])
model_b.summary()

## Step 4: sanity-check before training

In [ ]:
loss0, acc0 = model_b.evaluate(xb_train[:2000], yb_train[:2000], verbose=0)
print(f"untrained: loss={loss0:.3f}  accuracy={acc0:.3f}")
# expect loss near ln(2) = 0.693 and accuracy near 0.5

**Question:** why ln 2?

<details><summary>Show the answer</summary>

An untrained network outputs p ≈ 0.5 for every review, and the binary cross-entropy of a 0.5 prediction is −ln 0.5 = ln 2 ≈ 0.693, whatever the true label. The chance baseline for accuracy is 50% because the classes are balanced.

</details>

## Step 5: train with a validation split

`validation_split=0.4` keeps the last 10,000 training reviews for validation. Keras shuffled the reviews when loading them, so the split is fair.

In [ ]:
hb = model_b.fit(xb_train, yb_train, epochs=20, batch_size=512,
                 validation_split=0.4)       # 10,000 for validation

## Step 6: learning curves, and the best epoch

In [ ]:
h = hb.history
ep = range(1, len(h["loss"]) + 1)
best_epoch = int(np.argmin(h["val_loss"])) + 1
print("lowest validation loss at epoch", best_epoch)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ep, h["loss"], "o-", label="training")
ax1.plot(ep, h["val_loss"], "s--", label="validation")
ax1.axvline(best_epoch, color="grey", lw=0.8)
ax1.set(xlabel="epoch", ylabel="loss", title="Loss")
ax2.plot(ep, h["accuracy"], "o-", label="training")
ax2.plot(ep, h["val_accuracy"], "s--", label="validation")
ax2.set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
for ax in (ax1, ax2):
    ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Question:** what happens after the best epoch?

<details><summary>Show the answer</summary>

Training loss keeps falling towards zero while validation loss rises: the network is overfitting (Section 7). Notice that validation accuracy barely moves while validation loss climbs: the model is becoming more and more confident, including on the reviews it gets wrong. That is why we choose the epoch by loss, not accuracy.

</details>

## Step 7: choose the decision threshold on validation data

The sigmoid gives a probability *p*; we still have to choose the threshold *t* above which a review counts as positive. **0.5 assumes both kinds of error cost the same.** For a fault detector, a missed fault usually costs far more than a false alarm, so you would lower *t*.

Choose *t* on validation data, never on the test set. We retrain a fresh model for `best_epoch` epochs on the first 15,000 reviews and score the remaining 10,000. Read "positive" as "fault" for the engineering version.

In [ ]:
x_tr, y_tr = xb_train[:15000], yb_train[:15000]
x_val, y_val = xb_train[15000:], yb_train[15000:]

def build_imdb(units=16):
    model = keras.Sequential([
        keras.Input(shape=(10000,)),
        layers.Dense(units, activation="relu"),
        layers.Dense(units, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", metrics=["accuracy"],
                  loss="binary_crossentropy")
    return model

keras.utils.set_random_seed(0)
m_val = build_imdb()
m_val.fit(x_tr, y_tr, epochs=best_epoch, batch_size=512, verbose=0)
p_val = m_val.predict(x_val, verbose=0)[:, 0]

print(" t    accuracy  false alarms  missed positives")
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = p_val > t
    acc = np.mean(pred == y_val)
    false_alarm = np.mean(pred[y_val == 0])     # negatives called positive
    missed = np.mean(~pred[y_val == 1])         # positives called negative
    print(f"{t:.1f}   {acc:.3f}     {false_alarm:.3f}         {missed:.3f}")

plt.figure(figsize=(7, 3.5))
plt.hist(p_val[y_val == 0], bins=50, alpha=0.6, label="truly negative")
plt.hist(p_val[y_val == 1], bins=50, alpha=0.6, label="truly positive")
plt.xlabel("predicted probability of positive"); plt.ylabel("reviews"); plt.legend()
plt.show()

In [ ]:
threshold = 0.5   # change this after reading the table, e.g. to keep missed positives below 5%

## Step 8: final model, and the test set once

Retrain on **all 25,000** training reviews for `best_epoch` epochs, then evaluate on the test set, once. The reference run in the lecture gave about **88%** test accuracy and a test loss of 0.29.

In [ ]:
keras.utils.set_random_seed(0)
final = build_imdb()
final.fit(xb_train, yb_train, epochs=best_epoch, batch_size=512, verbose=0)

test_loss, test_acc = final.evaluate(xb_test, yb_test, verbose=0)
p_test = final.predict(xb_test, verbose=0)[:, 0]
print(f"test loss {test_loss:.3f}, test accuracy at t = 0.5: {test_acc:.3f}")
print(f"test accuracy at your threshold t = {threshold}: {np.mean((p_test > threshold) == yb_test):.3f}")

## Step 9: read the most confident mistakes

Bag of words cannot see word order, so watch for negation ("not good"), sarcasm, and reviews that praise one thing and condemn another.

In [ ]:
wrong = np.flatnonzero((p_test > 0.5) != yb_test)
conf = np.abs(p_test[wrong] - 0.5)             # distance from the threshold
for i in wrong[np.argsort(-conf)][:3]:
    truth = "positive" if yb_test[i] else "negative"
    print(f"true {truth}, predicted p(positive) = {p_test[i]:.2f}")
    print(decode(seq_test[i])[:600], "...\n")

---
# Part 2: overfitting and how to avoid it (Sections 7 and 8)

## A. Model size

The slide code from Section 8: the same architecture with 4, 16 and 512 units per hidden layer.

In [ ]:
val_loss = {}
for units in [4, 16, 512]:   # ~40k, 160k, 5.4M weights
    keras.utils.set_random_seed(0)
    h = build_imdb(units).fit(xb_train, yb_train, epochs=20,
        batch_size=512, validation_split=0.4, verbose=0)
    val_loss[units] = h.history["val_loss"]

plt.figure(figsize=(7, 4))
for units, vl in val_loss.items():
    plt.plot(range(1, len(vl) + 1), vl, label=f"{units} units")
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**Question:** which model overfits first, and which reaches the lowest validation loss?

<details><summary>Show the answer</summary>

The 512-unit model overfits almost immediately and its validation loss is noisy. The 4-unit model overfits latest and slowest. The best model is the smallest one whose validation loss turns up only after several epochs; read the minima off your plot.

</details>

## B. Dropout, L2 and early stopping

Also from the Section 8 slides: a plain model, then dropout, then dropout plus L2, all with early stopping. Each line prints the dropout rate, the L2 strength, the lowest validation loss and its epoch.

In [ ]:
def build_reg(rate, l2):
    reg = regularizers.l2(l2)
    model = keras.Sequential([
        keras.Input(shape=(10000,)),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dropout(rate),
        layers.Dense(16, activation="relu", kernel_regularizer=reg),
        layers.Dropout(rate),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile("adam", "binary_crossentropy", metrics=["accuracy"])
    return model

for rate, l2 in [(0.0, 0.0), (0.5, 0.0), (0.5, 1e-3)]:
    keras.utils.set_random_seed(0)
    stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True)
    h = build_reg(rate, l2).fit(
        xb_train, yb_train, epochs=40, batch_size=512,
        validation_split=0.4, callbacks=[stop], verbose=0).history
    print(rate, l2, min(h["val_loss"]), np.argmin(h["val_loss"]) + 1)

**Question:** with dropout and L2 the training loss ends higher. Is that a problem?

<details><summary>Show the answer</summary>

No. The training loss now includes the L2 penalty and is measured with dropout switched on. Only validation loss tells you about new data. A useful regulariser lowers the **minimum** validation loss, not just delays it. Repeat with a few seeds before trusting a small difference.

</details>

## Exercises

1. **Counts instead of presence:** change `multi_hot` so position *i* holds how often word *i* appears. Does accuracy change?
2. **Vocabulary size:** reload with `num_words=2000` and `num_words=20000`. What happens to accuracy, memory and the number of parameters?
3. **No hidden layer:** `Dense(1, activation="sigmoid")` straight on the 10,000 inputs is logistic regression. How far behind the MLP is it? Its 10,000 weights are readable: decode the words with the largest positive and negative weights.
4. **A costly miss:** suppose a missed negative review costs five times a false alarm. Pick the threshold on validation data that minimises the total cost, then report the test accuracy at that threshold.